# Lab 5 — Notebook 01: Understanding the PyTorch Training Loop

**Core duration: 35 minutes.** Dataset: the same 1,797 handwritten digits as Lab 4.

By the end, identify tensors, layers, logits (— raw, unnormalized scores before a probability transformation), loss, automatic gradients, and optimizer updates. Connect the NumPy gradient-descent loop in Lab 3 to PyTorch, then explain what Ultralytics automates.

**Route:** tensors 6 min → network 7 min → training 12 min → mistakes/bridge 10 min. Times include core questions. Architecture experiments are optional. All worked examples run without solving TODO cells.


## 1. Load the same digits and create tensors (6 min)

Each image has 64 pixels with known range 0–16. Dividing by the fixed maximum 16 is not fitting a scaler to the dataset. Use float32 features and int64 class labels.

Keep the same stratified 64% training / 16% validation / 20% reserved partition as Lab 4. **Lab 4 already revealed this test set:** it is no longer a fresh independent test for a course-wide model competition. This notebook uses validation only and makes no new final-test claim.


In [ ]:
from pathlib import Path
# Support notebooks started in lab05/ or in the repository root.
LAB_DIR = Path.cwd().resolve()
if (LAB_DIR / "lab05" / "data" / "digits.csv").is_file():
    LAB_DIR = LAB_DIR / "lab05"
if not (LAB_DIR / "data" / "digits.csv").is_file():
    raise FileNotFoundError("Run this notebook in lab05/ or the repository root; keep data/ and scripts/ in place.")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from sklearn.model_selection import train_test_split

np.random.seed(42)
torch.manual_seed(42)
torch.set_num_threads(2)  # polite default on shared CPU laboratory computers

df = pd.read_csv(LAB_DIR / 'data/digits.csv')
X = df[[f'pixel_{i}' for i in range(64)]].to_numpy(dtype=np.float32) / 16.0
y = df['target'].to_numpy(dtype=np.int64)

# 64% train / 16% validation / 20% reserved test (we use val only in this notebook)
X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(
    X_dev, y_dev, test_size=0.20, random_state=42, stratify=y_dev)
X_train_t = torch.tensor(X_train)
y_train_t = torch.tensor(y_train)
X_val_t = torch.tensor(X_val)
y_val_t = torch.tensor(y_val)
print('train:',len(X_train),'validation:',len(X_val),'reserved test:',len(X_test))
print('Input tensor:',X_train_t.shape, X_train_t.dtype)
print('Target tensor:',y_train_t.shape, y_train_t.dtype)
assert X_train_t.ndim == 2 and X_train_t.shape[1] == 64
assert y_train_t.ndim == 1 and y_train_t.dtype == torch.int64
assert X_train_t.dtype == torch.float32


In [ ]:
fig, axes=plt.subplots(1,5,figsize=(10,2.2))
for i,ax in enumerate(axes):
    ax.imshow(X_train[i].reshape(8,8),cmap='gray_r',vmin=0,vmax=1)
    ax.set_title(f'Label: {y_train[i]}')
    ax.axis('off')
plt.tight_layout()

**Exercise 1. Tensors and image data**  
Inspect an example's 64 features, reshape the example to `8 × 8`, and explain how a pixel value relates to the image. Use `# TODO` in the next cell.

In [ ]:
# TODO: Print the first 8 features for image 0.
# TODO: Print the minimum and maximum feature values.
# TODO: Explain why we divided pixel intensities by 16.

## 2. Construct a neural network (7 min)

A fully connected network is a stack of transformations. A hidden layer uses a nonlinear **ReLU activation**, and the final layer produces 10 raw class scores called **logits**.

\begin{equation*}
\mathbf{h} = \operatorname{ReLU}(W_1\mathbf{x}+\mathbf{b}_1), \qquad
\mathbf{z} = W_2\mathbf{h}+\mathbf{b}_2
\end{equation*}

`CrossEntropyLoss` accepts logits directly—**do not** apply `softmax` before passing scores to this loss.
**Shape check:** `(batch, 64)` → `(batch, 32)` → `(batch, 10)`. The target has shape `(batch,)`. `argmax(dim=1)` chooses the highest-scoring class.


In [ ]:
from torch import nn
model=nn.Sequential(
    nn.Linear(64,32),
    nn.ReLU(),
    nn.Linear(32,10)
)
loss_fn=nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(model.parameters(),lr=0.01)
print(model)
with torch.no_grad():
    sample_scores=model(X_train_t[:2])
print('Scores shape:',tuple(sample_scores.shape))
print('Predictions before training:',sample_scores.argmax(dim=1).tolist())

**Exercise 2. Before training**  
Why are the initial digit predictions mostly unreliable? How many trainable parameters does this network have?

In [ ]:
# TODO: Use sum(p.numel() for p in model.parameters()) to count parameters.
# TODO: Write your explanation in a Markdown cell.

## 3. Train the network (12 min)

The key steps are **forward pass → calculate loss → clear old gradients → backpropagate → update weights**. This should remind you of Lab 3.

\begin{equation*}
\theta_{t+1} = \theta_t - \alpha\nabla_{\theta}L(\theta_t)
\end{equation*}

Adam is an optimizer that adapts its parameter updates; the formula above describes the simpler gradient-descent idea. We train using the **whole training split per epoch** for clarity.
**Core question:** Point to the five steps below. Why do we clear gradients on every update? `backward()` computes gradients; `step()` changes parameters. `eval()` changes layer behavior, while `no_grad()` disables gradient recording. They serve different purposes.

A full-batch update is one iteration and one epoch here. With mini-batches, an epoch contains several updates. Rerunning this cell continues training the same model; rerun the construction cell first for a fresh experiment.


In [ ]:
history={'train_loss':[],'val_loss':[],'train_acc':[],'val_acc':[]}
for epoch in range(30):
    model.train()
    scores=model(X_train_t)
    loss=loss_fn(scores,y_train_t)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        train_scores=model(X_train_t)
        val_scores=model(X_val_t)
        history['train_loss'].append(loss_fn(train_scores,y_train_t).item())
        history['val_loss'].append(loss_fn(val_scores,y_val_t).item())
        history['train_acc'].append((train_scores.argmax(1)==y_train_t).float().mean().item())
        history['val_acc'].append((val_scores.argmax(1)==y_val_t).float().mean().item())
    if (epoch+1)%10==0:
        print(f'Epoch {epoch+1:2d}: train accuracy {history["train_acc"][-1]:.1%}, '
              f'validation accuracy {history["val_acc"][-1]:.1%}')

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(10,3.5))
epochs = np.arange(1, len(history['train_loss']) + 1)
ax[0].plot(epochs, history['train_loss'],label='Train')
ax[0].plot(epochs, history['val_loss'],label='Validation')
ax[0].set(title='Cross-entropy loss',xlabel='Epoch',ylabel='Loss')
ax[0].legend()
ax[1].plot(epochs, history['train_acc'],label='Train')
ax[1].plot(epochs, history['val_acc'],label='Validation')
ax[1].set(title='Digit classification accuracy',xlabel='Epoch',ylabel='Accuracy',ylim=(0,1.02))
ax[1].legend()
plt.tight_layout()
plt.show()


**Exercise 3. Learning curves**  
Identify an epoch when the network has clearly improved. Explain the difference between loss and accuracy. Is validation accuracy always guaranteed to increase?

In [ ]:
# TODO: Print the final training and validation loss.
# TODO: Print final validation accuracy.
# TODO: Write 2–3 sentences about the curves.

## 4. Make predictions and investigate mistakes (10 min, including bridge discussion)

The network outputs 10 scores; the index of the largest score is the predicted digit. Focus on examples that were not used to update its weights.


In [ ]:
model.eval()
with torch.no_grad():
    val_pred=model(X_val_t).argmax(dim=1).numpy()
wrong=np.where(val_pred != y_val)[0]
print('Validation mistakes:',len(wrong),'of',len(y_val))
if len(wrong) > 0:
    fig,axes=plt.subplots(1,min(5,len(wrong)),figsize=(10,2.5))
    axes=np.atleast_1d(axes)
    for ax,i in zip(axes,wrong[:5]):
        ax.imshow(X_val[i].reshape(8,8),cmap='gray_r',vmin=0,vmax=1)
        ax.set_title(f'True {y_val[i]} / Pred {val_pred[i]}')
        ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print("No mistakes found in this validation split.")


## Extension — Change one setting (optional, 7 min)

Change the hidden width to 16 or 64, **or** change the learning rate. Set `torch.manual_seed(42)`, construct a fresh model **and optimizer**, reset history, then train. Keep the split and epoch budget fixed. Compare validation results; do not inspect test data.

Record the setting, validation accuracy, and one explanation. A neural network is not guaranteed to outperform the classical models in Lab 4.


In [ ]:
# TODO: Try one change to architecture OR learning rate.
# TODO: Train a fresh model using the loop above and compare its validation accuracy.
# TODO: Briefly explain the result in a Markdown cell.

### Core bridge — What changes when using Ultralytics?

| Lab 3: NumPy | This notebook: PyTorch | Next: Ultralytics |
|---|---|---|
| Write prediction formula | Define layers and forward pass | Load a pretrained detection architecture |
| Write gradient expressions | `loss.backward()` computes gradients | Training method manages backpropagation |
| Update `w` and `b` | Optimizer updates parameters | Training method manages optimizer and batches |
| Inspect errors | Inspect loss and validation accuracy | Inspect detection metrics and images |

**Core reflection:** Which responsibilities remain yours when a library manages the training loop? Explain data collection, labels, independent evaluation, and failure analysis.

**Deliverable:** Learning curves, an explanation of the update steps, and an inspected validation error (or a note that none occurred).

**Next:** Notebook 02 explains what makes object detection evaluation different from digit classification.
